# 01. EDA
지수 일봉 데이터 점검과 목표 시계열(m5, m20, yz20) 모양 확인.

`REFRESH=True` 면 여기서 바로 다운로드·정제·지표 계산까지 수행합니다 (터미널에서 `python run.py fetch && python run.py build` 와 동일).

In [ ]:
import sys, pathlib, warnings
cwd = pathlib.Path.cwd()          # 커널 cwd 가 notebooks/ 든 프로젝트 루트든 상관없이 루트를 찾는다
ROOT = next(p for p in [cwd, *cwd.parents] if (p / "configs").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 100

In [ ]:
REFRESH = False       # True: 데이터를 새로 받고 다시 계산
TICKERS = ["SP500", "NASDAQ", "KOSPI"]

In [ ]:
from src.common import DATA_PROCESSED
if REFRESH or not (DATA_PROCESSED / "KOSPI.parquet").exists():
    from src.data.fetch import fetch_all
    from src.data.clean import clean_all
    from src.data.process import build_processed
    fetch_all(); clean_all(); build_processed()

from src.data.process import load_processed
data = {k: load_processed(k) for k in TICKERS}
pd.DataFrame({k: {"start": d.index[0].date(), "end": d.index[-1].date(), "rows": len(d)} for k, d in data.items()}).T

In [ ]:
# 정제 리포트: 평탄 봉(O=H=L=C), 이상치 일간수익률
import json
rep = json.load(open(ROOT / "data/interim/clean_report.json", encoding="utf-8"))
pd.DataFrame({k: {a: (len(b) if isinstance(b, list) else b) for a, b in v.items()} for k, v in rep.items()})

In [ ]:
for k, d in data.items():
    print(k)
    display(d[["m5", "m20", "yz20"]].describe().round(4))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for k, d in data.items():
    for ax, s in zip(axes, ["m5", "m20", "yz20"]):
        d[s].plot(ax=ax, lw=0.7, label=k); ax.set_ylabel(s)
axes[0].legend(); axes[0].set_title("Target series by index")
fig.tight_layout()